# Обучение модели регрессии для IC50

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.svm import SVR

from scipy.stats import loguniform

from catboost import CatBoostRegressor

import tqdm
from tqdm.auto import tqdm

import pickle

In [2]:
RANDOM_STATE = 22

Загружаем данные. Признаки уже стандартизированы, а таргет логарифмирован.

In [3]:
train_df = pd.read_csv("../../data/processed/ic50_train.csv")
test_df = pd.read_csv("../../data/processed/ic50_test.csv")

In [4]:
X_train = train_df.drop(columns=["IC50, mM"])
y_train = train_df["IC50, mM"]

X_test = test_df.drop(columns=["IC50, mM"])
y_test = test_df["IC50, mM"]

print(f"В обучающей выборке содержится {X_train.shape[0]} строк и {X_train.shape[1]} признаков")
print(f"В тестовой выборке содержится {X_test.shape[0]} строк и {X_test.shape[1]} признаков")

В обучающей выборке содержится 798 строк и 30 признаков
В тестовой выборке содержится 200 строк и 30 признаков


Зададим параметры сетки для подбора параметров четырех разных моделей регрессии.

In [ ]:
param_distributions = {
    "RidgeRegression": {
        'model': Ridge(),
        'params': {
            'alpha': loguniform(1e-2, 1e3)
        }
    },
    "GradientBoosting": {
        'model': GradientBoostingRegressor(random_state=RANDOM_STATE),
        'params': {
            'n_estimators': [100, 150, 200],
            'learning_rate': [0.01, 0.05, 0.1, 0.2],
            'max_depth': [3, 4, 5, 6],
            'subsample': [0.7, 0.8, 1.0],
            'min_samples_leaf': [1, 3, 5]
        }
    },
    "CatBoost": {
        "model": CatBoostRegressor(
            allow_writing_files=False,
            loss_function="RMSE", 
            verbose=False, 
            thread_count=1, 
            random_seed=RANDOM_STATE, 
            boosting_type="Plain",
            grow_policy="Lossguide",
            leaf_estimation_iterations=1
        ),
        "params": {
            "iterations": [150, 300, 500],
            "learning_rate": np.logspace(-2, -1, 5),
            "depth": [4, 5, 6, 7],
            "l2_leaf_reg": [3, 5, 10, 15],
            "max_leaves": [30, 50],
        }
    },
    "SVR": {
        "model": SVR(kernel="rbf"),
        "params": {
            "C": np.logspace(-2, 3, 20),  # От 0.01 до 1000
            "gamma": np.logspace(-4, 1, 20),  # От 0.0001 до 10
            "epsilon": [0.01, 0.05, 0.1, 0.2],  # Чувствительность к ошибкам
        }
    }
}

In [6]:
best_models = {}
for name, config in tqdm(param_distributions.items()):
    print(f"\n  Модель: {name}")
    print("="*50)
    
    grid_search = RandomizedSearchCV(
        random_state=RANDOM_STATE,
        estimator=config['model'],
        param_distributions=config['params'],
        cv=5,
        scoring='r2',
        n_iter=20,
        n_jobs=-1
    )
    
    grid_search.fit(X_train, y_train)
    
    # Оцениваем лучшую модель на тестовой выборке
    best_model = grid_search.best_estimator_
    best_models[name] = best_model
    
    preds = best_model.predict(X_test)
    r2 = r2_score(y_test, preds)

    # MAE оценим на данных в реальном масштабе
    mae = mean_absolute_error(10**(-y_test), 10**(-preds))
    print(f"    Лучшие параметры: {grid_search.best_params_}")
    print(f"    Тестовый R²: {r2:.4f}")
    print(f"    Тестовый MAE: {mae:.4f}")

  0%|          | 0/4 [00:00<?, ?it/s]


  Модель: RidgeRegression
    Лучшие параметры: {'alpha': np.float64(32.33623262496523)}
    Тестовый R²: 0.2384
    Тестовый MAE: 245.7523

  Модель: GradientBoosting
    Лучшие параметры: {'subsample': 0.7, 'n_estimators': 150, 'min_samples_leaf': 3, 'max_depth': 5, 'learning_rate': 0.05}
    Тестовый R²: 0.5514
    Тестовый MAE: 195.4543

  Модель: CatBoost
    Лучшие параметры: {'max_leaves': 50, 'learning_rate': np.float64(0.05623413251903491), 'l2_leaf_reg': 3, 'iterations': 150, 'depth': 5}
    Тестовый R²: 0.5351
    Тестовый MAE: 206.3830

  Модель: SVR
    Лучшие параметры: {'gamma': np.float64(0.023357214690901212), 'epsilon': 0.2, 'C': np.float64(4.281332398719392)}
    Тестовый R²: 0.5080
    Тестовый MAE: 201.3645


Лучшей моделью показал себя GradientBoosting с параметрами 

{'subsample': 0.7,<br>
'n_estimators': 150,<br>
'min_samples_leaf': 3,<br>
'max_depth': 5,<br>
'learning_rate': 0.05}. 

Сериализуем лучшую модель и сохраним в файл.

In [7]:
with open('../../models/model_ic50_regression.pkl', 'wb') as output:
    pickle.dump(best_models["GradientBoosting"], output)